# 🚀 **AIE4ML Tutorial: From QKeras → hls4ml → AMD AI Engine**

This tutorial shows how to:

- Build a small **quantized QKeras** model
- Generate AIE firmware with **AIE4ML**, using **hls4ml** as the frontend
- Let the AIE4ML compiler choose each layer's split, optimizing for **`'resource'`** or **`'performance'`**
- Check the result **bit-exact against QKeras** in x86 or aie simulation
- Read the **AIE simulation report**

---

---

## Environment Setup

```bash
conda create -n aie4ml python=3.11 -y && conda activate aie4ml
pip install "aie4ml[onnx,hls4ml]" qkeras-v3 "tensorflow~=2.16.0" pyparsing ipykernel
# Keras 3 / QKeras v3 need hls4ml from upstream until its 1.4 release:
pip install "hls4ml @ git+https://github.com/fastmachinelearning/hls4ml.git@a2abb4d22e7762a870cd46ccf3e07ff6a5622ed5"
```

# 1️⃣ Setup & Imports

In [ ]:
import numpy as np
import keras
from keras.layers import Flatten, Input, MaxPooling2D
from keras.models import Sequential

from qkeras import QActivation, QConv2D, QDense, quantized_bits, quantized_relu

import hls4ml

keras.utils.set_random_seed(42)

# 2️⃣ Build a Small QKeras CNN

- **Convolutions:** AIE4ML supports standard, grouped, depthwise, and 1×1 convolutions, with padding and strides. How efficient a convolution is depends on its shape.
- **Fusion:** A ReLU or a 2×2 max pool is fused in the compute kernels
- **Precision:** Put a quantizer after every layer and convert with `bit_exact=True`, so hls4ml derives the exact precision of every tensor.


In [ ]:
H, W, C = 16, 16, 3
CLASSES = 10
q8 = quantized_bits(8, 0, alpha=1)


def conv(filters, name):
    return [QConv2D(filters, 3, padding='same', kernel_quantizer=q8, bias_quantizer=q8, name=name),
            QActivation(quantized_relu(8, 2), name=f'{name}_relu')]


model = Sequential([
    Input((H, W, C)),
    QActivation(quantized_bits(8, 2), name='input_quant'),
    *conv(16, 'conv1'),
    *conv(16, 'conv2'),
    MaxPooling2D(2, name='pool1'),
    *conv(32, 'conv3'),
    MaxPooling2D(2, name='pool2'),
    Flatten(name='flatten'),
    QDense(32, kernel_quantizer=q8, bias_quantizer=q8, name='fc1'),
    QActivation(quantized_relu(8, 2), name='fc1_relu'),
    QDense(CLASSES, kernel_quantizer=q8, bias_quantizer=q8, name='fc2'),
    QActivation(quantized_bits(8, 2), name='output_quant'),
])
model.summary()


# 3️⃣ Convert to the AI Engine

Conv2D and Dense/matmul layers can be split across multiple AI Engine tiles. AIE4ML chooses the splits for the full model and keeps layer-to-layer transfers direct where possible. `optimize` controls the objective:

* `'resource'` (default) — uses the fewest tiles needed.
* `'performance'` — the lowest latency among the designs with the best throughput (output interval), within `max_tiles` (default: the whole array).

To fix a layer's split or placement manually, see Tutorial 2; the compiler keeps your choices and optimizes the rest.

> ⚠️ Large `max_tiles` values can increase optimization time for multi-layer or branching graphs. Most searches complete within seconds, while more complex graphs may take longer.

AI Engine parallelism is shaped by each kernel's vector width, which sets a minimum partition size. Very small models like the example may therefore be unable to utilize much of the array, while larger models provide more opportunities for efficient splitting.


In [ ]:
# `batch_size` is how many samples one graph call processes. Keras shapes have no batch axis, so hls4ml takes it here
# Keep 1 for the lowest latency; Conv2D layers currently can process one sample per call.
# For Dense, MatMul, LayerNorm, Softmax and Add, a larger batch is supported and can raise throughput.
BATCH_SIZE = 1

# Graph calls to simulate, each on BATCH_SIZE new samples
ITERS = 10
# Maximum tiles to use for the AIE graph.
MAX_TILES = 16


# Target platform examples:
#   AIE1:     xilinx_vck190_base, xcvp2802-vsva5601-2mhp-e-s
#   AIE-ML:   xilinx_vek280_base
#   AIE-MLv2: vek385_base
PLATFORM = "xilinx_vek280_base"

cfg = hls4ml.utils.config_from_keras_model(model, granularity='name')

aie_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=cfg,
    output_dir='proj_aie',
    backend='aie',
    project_name='proj_aie',
    batch_size=BATCH_SIZE,
    iterations=ITERS,
    part=PLATFORM,
    bit_exact=True,
    optimize='performance',
    max_tiles=MAX_TILES,
)
aie_model.write()

In [ ]:
from aie4ml.report import layout, report

# After project emission, you can optionally inspect its AIE array layout.
layout('proj_aie')

## 4️⃣ Bit-Exact Check against QKeras (x86)

The x86 simulator runs the AIE kernels functionally and compares every inference against the QKeras reference. This provides a fast way to validate bit-exact output before running AIE simulation.

> ⚠️ In rare cases, x86 simulation may differ from `aiesim`, so `aiesim` should still be used as the final validation step.

In [ ]:
aie_model.compile()

x = np.random.default_rng(0).uniform(-4, 4, size=(ITERS * BATCH_SIZE, H, W, C)).astype(np.float32)
y_qkeras = model.predict(x, verbose=0)


def compare(aie4ml_model, sim_mode='x86'):
    y_aie = np.asarray(aie4ml_model.predict(x.reshape(ITERS, BATCH_SIZE, H, W, C), simulator=sim_mode)).reshape(y_qkeras.shape)
    print('max |aie4ml - QKeras| :', np.abs(y_aie - y_qkeras).max())


compare(aie_model)

# 5️⃣ Build and Simulate on the AI Engine

`build()` compiles the AIE graph; `jobs=N` compiles `N` kernels at once (default: 8), which helps a design split across many tiles. `simulator='aie'` runs the cycle-accurate AIE simulator with per-kernel profiling.

In [ ]:
aie_model.build(jobs=8)
compare(aie_model, sim_mode='aie')

# 6️⃣ View the AIE Project Report

`report()` gathers everything the AMD toolchain left in the project:

* **Latency** — one sample end to end, split into compute (the critical path) and data movement
* **Throughput** — steady-state output interval, GOP/s, per-PLIO bandwidth
* **Per-stage cost** — cycles, utilisation and program memory for every kernel
* **Resources** — AIE tiles, memtile buffers, memory high-water marks, and the split the compiler chose

It accepts either a model or a project directory. The same output in a terminal: `aie4ml-report proj_aie`.

In [ ]:
from aie4ml.report import report
report('proj_aie')

# 🎉 Tutorial Complete!